# Data Pipeline & Exploratory Analysis

Analyzing southern pine forest growth using USDA Forest Inventory & Analysis (FIA) public data across Georgia (GA), Alabama (AL), and South Carolina (SC).

**Run this notebook from the `notebooks/` directory.**

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Southern pine species (USDA FIA species codes, all ModRel=1)
# Source: FIA Species Code reference list
SOUTHERN_PINE_SPECIES = {
    110: "Shortleaf pine",   # Pinus echinata
    111: "Slash pine",       # Pinus elliottii
    121: "Longleaf pine",    # Pinus palustris
    131: "Loblolly pine",    # Pinus taeda
}

STATES = ["GA", "AL", "SC"]
TABLES = ["TREE", "PLOT", "COND"]

DATA_DIR = "../data"

## 1. Load Raw Data

In [2]:
TREE_COLS = ["CN", "PLT_CN", "CONDID", "SPCD", "STATUSCD", "DIA", "PREVDIA", "INVYR"]
PLOT_COLS = ["CN", "LAT", "LON", "ELEV", "REMPER", "MEASYEAR"]
COND_COLS = ["PLT_CN", "CONDID", "STDAGE"]

tree_frames, plot_frames, cond_frames = [], [], []

for state in STATES:
    tree_frames.append(pd.read_csv(os.path.join(DATA_DIR, f"{state}_TREE.csv"), usecols=TREE_COLS).assign(state=state))
    plot_frames.append(pd.read_csv(os.path.join(DATA_DIR, f"{state}_PLOT.csv"), usecols=PLOT_COLS).assign(state=state))
    cond_frames.append(pd.read_csv(os.path.join(DATA_DIR, f"{state}_COND.csv"), usecols=COND_COLS).assign(state=state))

print("Files read.")

Files read.


In [3]:
tree = pd.concat(tree_frames, ignore_index=True)
plot = pd.concat(plot_frames, ignore_index=True)
cond = pd.concat(cond_frames, ignore_index=True)

for name, df in [("TREE", tree), ("PLOT", plot), ("COND", cond)]:
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} columns")

tree.head()

TREE: 4,177,453 rows x 9 columns
PLOT: 161,563 rows x 7 columns
COND: 199,699 rows x 4 columns


,CN,PLT_CN,INVYR,CONDID,STATUSCD,SPCD,DIA,PREVDIA,state
0,158870927010854,158712463010854,1972,1,1,111.0,4.6,NaN,GA
1,158870928010854,158712463010854,1972,1,1,131.0,10.9,NaN,GA
2,158870929010854,158712463010854,1972,1,1,111.0,7.9,NaN,GA
3,158870930010854,158712463010854,1972,1,1,111.0,8.1,NaN,GA
4,158870931010854,158712463010854,1972,1,1,111.0,4.3,NaN,GA


In [4]:
for name, df in [("TREE", tree), ("PLOT", plot), ("COND", cond)]:
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} columns")

for name, df in [("TREE", tree), ("PLOT", plot), ("COND", cond)]:
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} columns")
    display(df.head())


TREE: 4,177,453 rows x 9 columns
PLOT: 161,563 rows x 7 columns
COND: 199,699 rows x 4 columns
TREE: 4,177,453 rows x 9 columns


,CN,PLT_CN,INVYR,CONDID,STATUSCD,SPCD,DIA,PREVDIA,state
0,158870927010854,158712463010854,1972,1,1,111.0,4.6,NaN,GA
1,158870928010854,158712463010854,1972,1,1,131.0,10.9,NaN,GA
2,158870929010854,158712463010854,1972,1,1,111.0,7.9,NaN,GA
3,158870930010854,158712463010854,1972,1,1,111.0,8.1,NaN,GA
4,158870931010854,158712463010854,1972,1,1,111.0,4.3,NaN,GA


PLOT: 161,563 rows x 7 columns


,CN,MEASYEAR,REMPER,LAT,LON,ELEV,state
0,76355732010478,2005.0,3.8,32.657528,-82.315483,280,GA
1,76355989010478,2005.0,5.9,32.056778,-81.835922,150,GA
2,76356069010478,2005.0,4.1,34.860071,-84.457850,2660,GA
3,76356282010478,2005.0,4.0,34.840699,-84.312903,2120,GA
4,76356452010478,2005.0,3.8,34.928583,-84.281292,1760,GA


COND: 199,699 rows x 4 columns


,PLT_CN,CONDID,STDAGE,state
0,158712463010854,1,15.0,GA
1,158712464010854,1,25.0,GA
2,158712465010854,1,35.0,GA
3,158712466010854,1,45.0,GA
4,158712467010854,1,5.0,GA


## 2. Explore TREE Table

In [5]:
# Dtypes and null counts for key columns
key_cols = ["SPCD", "STATUSCD", "DIA", "PREVDIA", "INVYR"]
print(tree[key_cols].dtypes)
print()
print("Null counts:")
print(tree[key_cols].isnull().sum())

SPCD        float64
STATUSCD      int64
DIA         float64
PREVDIA     float64
INVYR         int64
dtype: object

Null counts:
SPCD              0
STATUSCD          0
DIA           69590
PREVDIA     1081803
INVYR             0
dtype: int64


In [6]:
# Inventory years covered
tree["INVYR"].value_counts().sort_index()

INVYR
1968    140184
1972    316680
1978    157639
1982    285309
1986    136259
1989    262128
1990    128280
1993    174979
1997    317258
1998     34944
1999     63045
2000    248293
2001    102794
2002     72367
2003     72794
2004     76245
2005     93700
2006     78012
2007     79088
2008     73306
2009     77729
2010     78284
2011     80785
2012     82297
2013     80411
2014     81820
2015     80179
2016     80402
2017     83110
2018     80257
2019     80445
2020     77614
2021     78047
2022     73529
2023     71876
2024     77364
Name: count, dtype: int64

In [7]:
# Tree status: 1=live, 2=dead, 3=removed
tree["STATUSCD"].value_counts().sort_index()

STATUSCD
0      68935
1    3425672
2     557871
3     124975
Name: count, dtype: int64

In [8]:
# Top species by record count (before filtering)
tree["SPCD"].value_counts().head(20)

SPCD
131.0    1141750
611.0     405567
111.0     268493
316.0     215481
827.0     195150
694.0     179823
110.0     118848
621.0     101949
820.0      88079
802.0      85972
121.0      83535
491.0      77530
812.0      65350
693.0      58657
132.0      57920
400.0      56815
653.0      49908
762.0      46702
835.0      43467
711.0      43100
Name: count, dtype: int64

## 3. Explore PLOT Table

In [9]:
# Dtypes and null counts for key columns
plot_key_cols = ["LAT", "LON", "ELEV", "REMPER", "MEASYEAR"]
print(plot[plot_key_cols].dtypes)
print()
print("Null counts:")
print(plot[plot_key_cols].isnull().sum())

LAT         float64
LON         float64
ELEV          int64
REMPER      float64
MEASYEAR    float64
dtype: object

Null counts:
LAT           202
LON           202
ELEV            0
REMPER      32355
MEASYEAR     2442
dtype: int64


In [10]:
# Remeasurement period distribution — drives annual growth calculation
plot["REMPER"].value_counts().sort_index()

REMPER
0.0     204
0.1       2
0.2       1
0.3       2
0.4       8
       ... 
11.6     56
11.8      1
11.9      2
12.0      1
12.1      1
Name: count, Length: 121, dtype: int64

## 4. Explore COND Table

In [11]:
# Dtypes and null counts
print(cond[["STDAGE"]].dtypes)
print()
print("Null counts:")
print(cond[["STDAGE"]].isnull().sum())

STDAGE    float64
dtype: object

Null counts:
STDAGE    63252
dtype: int64


In [12]:
# Stand age distribution
cond["STDAGE"].describe()

count    136447.000000
mean         35.181316
std          26.439945
min           0.000000
25%          15.000000
50%          30.000000
75%          51.000000
max         184.000000
Name: STDAGE, dtype: float64